In [1]:
import torch
from pathlib import Path

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

from peft import PeftModel
from langchain_community.utilities import SQLDatabase
from langchain_core.runnables import RunnableLambda

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

CWD = Path.cwd()

if (CWD / "data").exists():
    BASE_DIR = CWD
else:
    BASE_DIR = CWD.parent

DB_PATH = BASE_DIR / "data" / "database" / "hospital.db"
ADAPTER_DIR = BASE_DIR / "models" / "qwen2.5-3b-medical-lora"

print("GPU:", torch.cuda.get_device_name(0))
print("Banco:", DB_PATH)
print("Adapter:", ADAPTER_DIR)

C:\Users\Diogo\anaconda3\envs\fiap_fase3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0912 21:39:59.511000 20032 site-packages\torch\utils\flop_counter.py:113] triton not found; flop counting will not work for triton kernels
C:\Users\Diogo\AppData\Local\Temp\ipykernel_20032\1025919737.py:11: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import SQLDatabase


GPU: NVIDIA GeForce RTX 3060
Banco: C:\Users\Diogo\tech-challenge-FIAP-fase3-assistente-medico\data\database\hospital.db
Adapter: C:\Users\Diogo\tech-challenge-FIAP-fase3-assistente-medico\models\qwen2.5-3b-medical-lora


In [2]:
db = SQLDatabase.from_uri(
    f"sqlite:///{DB_PATH.as_posix()}"
)

print(db.get_usable_table_names())

['pacientes']


In [3]:
import re

def buscar_paciente(id_paciente):

    id_paciente = id_paciente.upper().strip()

    if not re.fullmatch(r"PAC\d{3}", id_paciente):
        return "ID de paciente inválido."

    query = f"""
    SELECT
        id_paciente,
        idade,
        sexo,
        historico_familiar,
        resultado_exame,
        exames_pendentes,
        status
    FROM pacientes
    WHERE id_paciente = '{id_paciente}'
    """

    resultado = db.run(query)

    if not resultado:
        return "Paciente não encontrado."

    return resultado

In [4]:
print(buscar_paciente("PAC001"))

[('PAC001', 52, 'F', 'Sim', 'Achado suspeito', 'Ultrassonografia complementar', 'Em investigação')]


In [5]:
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(
    ADAPTER_DIR
)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    device_map="auto",
    dtype=torch.bfloat16
)

model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_DIR
)

model.eval()

print("Qwen Fine-Tuned carregado.")
print("Dispositivo:", model.device)

Loading weights: 100%|██████████| 434/434 [00:03<00:00, 127.64it/s]


Qwen Fine-Tuned carregado.
Dispositivo: cuda:0


In [6]:
def gerar_resposta_contextualizada(contexto_paciente, pergunta):

    messages = [
        {
            "role": "system",
            "content": """
Você é um assistente clínico de apoio à decisão médica.

Regras:
- Utilize os dados do paciente fornecidos no contexto.
- Não invente informações ausentes.
- Não emita diagnóstico definitivo.
- Não prescreva medicamentos.
- Não informe doses.
- Destaque exames pendentes quando relevantes.
- A decisão final deve permanecer com o profissional médico responsável.
""".strip()
        },
        {
            "role": "user",
            "content": f"""
DADOS DO PACIENTE:
{contexto_paciente}

PERGUNTA DO PROFISSIONAL:
{pergunta}

Responda considerando exclusivamente as informações disponíveis.
"""
        }
    ]

    texto = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        texto,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=220,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    novos_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        novos_tokens,
        skip_special_tokens=True
    )

In [9]:
import re

def buscar_paciente(id_paciente):

    id_paciente = id_paciente.upper().strip()

    if not re.fullmatch(r"PAC\d{3}", id_paciente):
        return "ID de paciente inválido."

    query = f"""
    SELECT
        id_paciente,
        idade,
        sexo,
        historico_familiar,
        resultado_exame,
        exames_pendentes,
        status
    FROM pacientes
    WHERE id_paciente = '{id_paciente}'
    """

    resultado = db.run(query)

    if not resultado:
        return "Paciente não encontrado."

    import ast

    registro = ast.literal_eval(resultado)[0]

    contexto = f"""
ID DO PACIENTE: {registro[0]}
IDADE: {registro[1]}
SEXO: {registro[2]}
HISTÓRICO FAMILIAR: {registro[3]}
RESULTADO DO EXAME JÁ REALIZADO: {registro[4]}
EXAME PENDENTE: {registro[5]}
STATUS ATUAL: {registro[6]}
""".strip()

    return contexto

In [10]:
resultado = chain_assistente.invoke({
    "id_paciente": "PAC001",
    "pergunta": (
        "Analise a situação atual deste paciente e informe "
        "quais pontos precisam de atenção."
    )
})

print("PACIENTE:")
print(resultado["id_paciente"])

print("\nDADOS CONSULTADOS:")
print(resultado["contexto_paciente"])

print("\nRESPOSTA DO ASSISTENTE:")
print(resultado["resposta"])

PACIENTE:
PAC001

DADOS CONSULTADOS:
ID DO PACIENTE: PAC001
IDADE: 52
SEXO: F
HISTÓRICO FAMILIAR: Sim
RESULTADO DO EXAME JÁ REALIZADO: Achado suspeito
EXAME PENDENTE: Ultrassonografia complementar
STATUS ATUAL: Em investigação

RESPOSTA DO ASSISTENTE:
A avaliação inicial indica que o paciente está em investigação por achado suspeito. Os principais pontos a serem considerados são:

- O paciente tem idade avançada (52 anos), fator de risco conhecido para alguns tipos de câncer.
- Há histórico familiar, o que aumenta o risco de certos tipos de doenças.
- O achado suspeito já foi identificado, indicando necessidade de investigação mais aprofundada.
- Exame ultrassonografia complementar está pendente, o que pode trazer mais informações importantes.
- Paciente está em investigação, o que significa que ainda não há diagnóstico definitivo.
- Informações sobre o tipo específico do achado suspeito não estão disponíveis.
- É importante considerar outros fatores de risco associados ao histórico fa